# Resolve Yahoo and FantasyPros Player Names

This notebook builds a Yahoo<->FantasyPros player crosswalk for a given season using the FantasyPros matching pipeline.

In [6]:
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 0 ns (started: 2026-07-26 17:00:39 -05:00)


In [28]:
from datetime import date
from pathlib import Path

import polars as pl

from nfl.fantasypros_fantasy import PipelineConfig, run_pipeline
from nfl.fantasypros_fantasy.storage.iceberg import IcebergCatalogConfig, persist_to_iceberg, IcebergNamespaceConfig
from nfl.yahoo_fantasy import YahooWarehouseClient

time: 0 ns (started: 2026-07-26 17:11:48 -05:00)


In [20]:
SEASON = 2025
WRITE_CROSSWALK_PARQUET = True
OUTPUT_DIR = Path('./output/yahoo_fantasypros_name_resolution')

# Optional: persist FantasyPros pipeline entities to Iceberg while building crosswalk.
PERSIST_TO_ICEBERG = True
ICEBERG_DRY_RUN = False

print('Season:', SEASON)
print('Persist to Iceberg:', PERSIST_TO_ICEBERG)
print('Iceberg dry run:', ICEBERG_DRY_RUN)
print('Output dir:', OUTPUT_DIR)

Season: 2025
Persist to Iceberg: True
Iceberg dry run: False
Output dir: output\yahoo_fantasypros_name_resolution
time: 0 ns (started: 2026-07-26 17:08:38 -05:00)


In [21]:
warehouse = YahooWarehouseClient.from_project_root()
_ = warehouse.ensure_registered()

preferred_player_tables = ['yahoo_common.player', 'yaml_common.player']
available_tables = set(warehouse.list_tables())
player_table_identifier = next((table for table in preferred_player_tables if table in available_tables), None)

if player_table_identifier is None:
    namespace_dirs = [p.name for p in warehouse.paths.warehouse_path.iterdir() if p.is_dir()]
    if namespace_dirs:
        _ = warehouse.ensure_registered(namespaces=namespace_dirs)
        available_tables = set(warehouse.list_tables())
        player_table_identifier = next((table for table in preferred_player_tables if table in available_tables), None)

if player_table_identifier is None:
    player_like_tables = sorted(table for table in available_tables if table.endswith('.player'))
    raise RuntimeError(
        'Could not locate Yahoo player table. Expected one of '
        f"{preferred_player_tables}. Available *.player tables: {player_like_tables}"
    )

print('Using player table:', player_table_identifier)
yahoo_player_df = warehouse.load_table(player_table_identifier)

# Keep a canonical shape expected by build_fp_yahoo_crosswalk.
yahoo_for_matching_df = (
    yahoo_player_df
    .select([
        pl.col('player_id').alias('yahoo_player_id'),
        pl.col('full_name'),
        pl.col('display_position'),
    ])
    .with_columns([
        pl.col('full_name').str.split(' ').list.get(0).fill_null('').alias('first_name'),
        pl.col('full_name').str.split(' ').list.last().fill_null('').alias('last_name'),
    ])
    .drop_nulls(['yahoo_player_id', 'full_name', 'display_position'])
    .unique(subset=['yahoo_player_id'], keep='first')
)

yahoo_players = yahoo_for_matching_df.to_dicts()
print('Yahoo players available for matching:', len(yahoo_players))

Using player table: yahoo_common.player
Yahoo players available for matching: 1383
time: 78 ms (started: 2026-07-26 17:08:40 -05:00)


In [22]:
storage_target = 'iceberg' if PERSIST_TO_ICEBERG else 'none'

catalog_db = (warehouse.paths.project_root / 'iceberg_catalog.db').resolve()
warehouse_dir = (warehouse.paths.project_root / 'warehouse').resolve()
iceberg_catalog = IcebergCatalogConfig(
    uri=f"sqlite:///{catalog_db.as_posix()}",
    warehouse=f"file://{warehouse_dir.as_posix()}",
)

result = run_pipeline(
    season=SEASON,
    sport='nfl',
    yahoo_players=yahoo_players,
    config=PipelineConfig(
        storage_target=storage_target,
        effective_date=date.today(),
        iceberg_dry_run=ICEBERG_DRY_RUN,
        iceberg_catalog=iceberg_catalog,
    ),
)

fp_players_df = result.frames.get('fp_player', pl.DataFrame())
crosswalk_df = result.frames.get('nfl_fp_yahoo_player_map', pl.DataFrame())

print('FP players:', fp_players_df.height)
print('Crosswalk matches:', crosswalk_df.height)


FP players: 5
Crosswalk matches: 5
time: 609 ms (started: 2026-07-26 17:08:41 -05:00)


In [23]:
if crosswalk_df.height == 0:
    print('No matches generated.')
else:
    fp_small_df = fp_players_df.select(['fp_player_id', 'full_name', 'position', 'team'])
    yahoo_small_df = yahoo_for_matching_df.select(['yahoo_player_id', 'full_name', 'display_position'])

    crosswalk_view_df = (
        crosswalk_df
        .join(fp_small_df, on='fp_player_id', how='left')
        .join(yahoo_small_df, on='yahoo_player_id', how='left', suffix='_yahoo')
        .rename({
            'full_name': 'fp_full_name',
            'full_name_yahoo': 'yahoo_full_name',
            'display_position': 'yahoo_display_position',
        })
        .select([
            'fp_player_id',
            'fp_full_name',
            'position',
            'team',
            'yahoo_player_id',
            'yahoo_full_name',
            'yahoo_display_position',
            'match_method',
            'matched_at',
        ])
        .sort(['match_method', 'fp_full_name'])
    )

    print('Match method counts:')
    print(crosswalk_view_df.group_by('match_method').len().sort('len', descending=True))
    print('Crosswalk preview:')
    print(crosswalk_view_df.head(50))

Match method counts:
shape: (1, 2)
┌──────────────┬─────┐
│ match_method ┆ len │
│ ---          ┆ --- │
│ str          ┆ u32 │
╞══════════════╪═════╡
│ exact        ┆ 5   │
└──────────────┴─────┘
Crosswalk preview:
shape: (5, 9)
┌────────────┬────────────┬──────────┬──────┬───┬────────────┬────────────┬────────────┬───────────┐
│ fp_player_ ┆ fp_full_na ┆ position ┆ team ┆ … ┆ yahoo_full ┆ yahoo_disp ┆ match_meth ┆ matched_a │
│ id         ┆ me         ┆ ---      ┆ ---  ┆   ┆ _name      ┆ lay_positi ┆ od         ┆ t         │
│ ---        ┆ ---        ┆ str      ┆ str  ┆   ┆ ---        ┆ on         ┆ ---        ┆ ---       │
│ str        ┆ str        ┆          ┆      ┆   ┆ str        ┆ ---        ┆ str        ┆ datetime[ │
│            ┆            ┆          ┆      ┆   ┆            ┆ str        ┆            ┆ μs, UTC]  │
╞════════════╪════════════╪══════════╪══════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ bijan-robi ┆ Bijan      ┆ RB       ┆ ATL  ┆ … ┆ Bijan      ┆ R

In [24]:
if fp_players_df.height == 0:
    print('No FantasyPros players loaded.')
else:
    matched_fp_ids_df = crosswalk_df.select(['fp_player_id']).unique() if crosswalk_df.height > 0 else pl.DataFrame({'fp_player_id': []})

    unmatched_fp_df = (
        fp_players_df
        .join(matched_fp_ids_df, on='fp_player_id', how='anti')
        .select(['fp_player_id', 'full_name', 'position', 'team'])
        .sort(['position', 'full_name'])
    )

    print('Unmatched FantasyPros players:', unmatched_fp_df.height)
    print(unmatched_fp_df.head(50))

Unmatched FantasyPros players: 0
shape: (0, 4)
┌──────────────┬───────────┬──────────┬──────┐
│ fp_player_id ┆ full_name ┆ position ┆ team │
│ ---          ┆ ---       ┆ ---      ┆ ---  │
│ str          ┆ str       ┆ str      ┆ str  │
╞══════════════╪═══════════╪══════════╪══════╡
└──────────────┴───────────┴──────────┴──────┘
time: 0 ns (started: 2026-07-26 17:08:43 -05:00)


In [25]:
if WRITE_CROSSWALK_PARQUET:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    crosswalk_path = OUTPUT_DIR / f'nfl_fp_yahoo_player_map_{SEASON}.parquet'
    unmatched_path = OUTPUT_DIR / f'nfl_fp_unmatched_{SEASON}.parquet'

    if crosswalk_df.height > 0:
        crosswalk_view_df.write_parquet(crosswalk_path)
        print('Wrote crosswalk parquet:', crosswalk_path)

    if fp_players_df.height > 0:
        unmatched_fp_df.write_parquet(unmatched_path)
        print('Wrote unmatched parquet:', unmatched_path)
else:
    print('Parquet output disabled.')

Wrote crosswalk parquet: output\yahoo_fantasypros_name_resolution\nfl_fp_yahoo_player_map_2025.parquet
Wrote unmatched parquet: output\yahoo_fantasypros_name_resolution\nfl_fp_unmatched_2025.parquet
time: 0 ns (started: 2026-07-26 17:08:46 -05:00)


In [26]:
# --- Backfill: collect FP players from prior seasons not seen in the primary season ---
BACKFILL_SEASONS = [2021, 2022, 2023, 2024]

primary_fp_ids = set(fp_players_df['fp_player_id'].to_list()) if fp_players_df.height > 0 else set()
backfill_frames: list[pl.DataFrame] = []

for season in BACKFILL_SEASONS:
    print(f'Fetching FP players for {season}...')
    season_result = run_pipeline(
        season=season,
        sport='nfl',
        config=PipelineConfig(
            storage_target='none',   # no Iceberg writes for backfill lookup
            effective_date=date.today(),
        ),
    )
    season_fp_df = season_result.frames.get('fp_player', pl.DataFrame())
    if season_fp_df.height > 0:
        # tag with season so we know where each player was last seen
        backfill_frames.append(season_fp_df.with_columns(pl.lit(season).alias('backfill_season')))
    print(f'  {season_fp_df.height} players')

if backfill_frames:
    backfill_all_df = pl.concat(backfill_frames, how='diagonal_relaxed')
    # keep most-recent season's row per fp_player_id, exclude those already in primary season
    backfill_new_df = (
        backfill_all_df
        .filter(~pl.col('fp_player_id').is_in(primary_fp_ids))
        .sort('backfill_season', descending=True)
        .unique(subset=['fp_player_id'], keep='first')
    )
    print(f'\nBackfill players not in {SEASON}: {backfill_new_df.height}')

    # build a combined crosswalk across primary + backfill players
    from nfl.fantasypros_fantasy.matching import build_fp_yahoo_crosswalk

    backfill_crosswalk_rows = build_fp_yahoo_crosswalk(
        fp_players=backfill_new_df.drop('backfill_season').to_dicts(),
        yahoo_players=yahoo_players,
    )
    backfill_crosswalk_df = pl.DataFrame(backfill_crosswalk_rows) if backfill_crosswalk_rows else pl.DataFrame()
    print(f'Backfill crosswalk matches: {backfill_crosswalk_df.height}')

    if backfill_crosswalk_df.height > 0:
        # exclude any yahoo_player_ids already matched in the primary crosswalk
        primary_yahoo_ids = set(crosswalk_df['yahoo_player_id'].to_list()) if crosswalk_df.height > 0 else set()
        backfill_crosswalk_df = backfill_crosswalk_df.filter(
            ~pl.col('yahoo_player_id').is_in(primary_yahoo_ids)
        )
        print(f'Backfill crosswalk matches (after excluding already-matched Yahoo IDs): {backfill_crosswalk_df.height}')
        print(backfill_crosswalk_df.head(30))
else:
    print('No backfill seasons produced results.')
    backfill_new_df = pl.DataFrame()
    backfill_crosswalk_df = pl.DataFrame()


Fetching FP players for 2021...
  5 players
Fetching FP players for 2022...
  5 players
Fetching FP players for 2023...
  5 players
Fetching FP players for 2024...
  5 players

Backfill players not in 2025: 12
Backfill crosswalk matches: 12
Backfill crosswalk matches (after excluding already-matched Yahoo IDs): 12
shape: (12, 4)
┌─────────────────────┬─────────────────┬──────────────┬────────────────────────────────┐
│ fp_player_id        ┆ yahoo_player_id ┆ match_method ┆ matched_at                     │
│ ---                 ┆ ---             ┆ ---          ┆ ---                            │
│ str                 ┆ i64             ┆ str          ┆ datetime[μs, UTC]              │
╞═════════════════════╪═════════════════╪══════════════╪════════════════════════════════╡
│ alvin-kamara        ┆ 30180           ┆ exact        ┆ 2026-07-26 22:08:47.832076 UTC │
│ austin-ekeler       ┆ 30423           ┆ exact        ┆ 2026-07-26 22:08:47.832076 UTC │
│ breece-hall         ┆ 33991          

In [16]:
# build a combined crosswalk across primary + backfill players
from nfl.fantasypros_fantasy.matching import build_fp_yahoo_crosswalk

backfill_crosswalk_rows = build_fp_yahoo_crosswalk(
    fp_players=backfill_new_df.drop('backfill_season').to_dicts(),
    yahoo_players=yahoo_players,
)
backfill_crosswalk_df = pl.DataFrame(backfill_crosswalk_rows) if backfill_crosswalk_rows else pl.DataFrame()
print(f'Backfill crosswalk matches: {backfill_crosswalk_df.height}')

if backfill_crosswalk_df.height > 0:
    # exclude any yahoo_player_ids already matched in the primary crosswalk
    primary_yahoo_ids = set(crosswalk_df['yahoo_player_id'].to_list()) if crosswalk_df.height > 0 else set()
    backfill_crosswalk_df = backfill_crosswalk_df.filter(
        ~pl.col('yahoo_player_id').is_in(primary_yahoo_ids)
    )
    print(f'Backfill crosswalk matches (after excluding already-matched Yahoo IDs): {backfill_crosswalk_df.height}')
    print(backfill_crosswalk_df.head(30))

Backfill crosswalk matches: 12
Backfill crosswalk matches (after excluding already-matched Yahoo IDs): 12
shape: (12, 4)
┌─────────────────────┬─────────────────┬──────────────┬────────────────────────────────┐
│ fp_player_id        ┆ yahoo_player_id ┆ match_method ┆ matched_at                     │
│ ---                 ┆ ---             ┆ ---          ┆ ---                            │
│ str                 ┆ i64             ┆ str          ┆ datetime[μs, UTC]              │
╞═════════════════════╪═════════════════╪══════════════╪════════════════════════════════╡
│ alvin-kamara        ┆ 30180           ┆ exact        ┆ 2026-07-26 22:07:17.598282 UTC │
│ austin-ekeler       ┆ 30423           ┆ exact        ┆ 2026-07-26 22:07:17.598282 UTC │
│ breece-hall         ┆ 33991           ┆ exact        ┆ 2026-07-26 22:07:17.598282 UTC │
│ christian-mccaffrey ┆ 30121           ┆ exact        ┆ 2026-07-26 22:07:17.598282 UTC │
│ cooper-kupp         ┆ 30182           ┆ exact        ┆ 2026-07-26 2

In [29]:
backfill_write_frames: dict[str, pl.DataFrame] = {}
if backfill_new_df.height > 0:
    backfill_write_frames['fp_player'] = backfill_new_df.drop('backfill_season')
    if backfill_crosswalk_df.height > 0:
        backfill_write_frames['nfl_fp_yahoo_player_map'] = backfill_crosswalk_df

backfill_iceberg_results = persist_to_iceberg(
    frames=backfill_write_frames,
    catalog_config=iceberg_catalog,
    namespace_config=IcebergNamespaceConfig(),
    dry_run=ICEBERG_DRY_RUN,
)

for r in backfill_iceberg_results:
    status = 'skipped (idempotent)' if r.skipped_by_idempotency else f'wrote {r.written_rows} rows'
    print(f'{r.table_identifier}: {status}')

fpcommon.fp_player: wrote 12 rows
fpnfl.fp_yahoo_player_map: wrote 12 rows
time: 141 ms (started: 2026-07-26 17:12:00 -05:00)
